# STPath — zero-shot generative foundation model

STPath (Huang et al., *npj Digital Medicine* 2025) is a generative
foundation model trained on 1,170 paired ST + H&E slides covering
17 organs and 38,984 genes. The published weights
(HuggingFace `tlhuang/STPath`) take **only** GigaPath features
and tile centroids — no reference Visium slide is required.

STPath currently leads HEST-Bench by +6.9 % Pearson over the next
best method.

**HuggingFace access** — both `prov-gigapath/prov-gigapath`
(used for tile features) and the implicit STPath weights live
behind HuggingFace gating. Request access on each model card,
then `huggingface-cli login` once on this machine. The cell that
calls `predict_expression(method='stpath')` will otherwise raise
`GatedRepoError`.

## Environment & demo data

In [ ]:
import warnings, os
warnings.filterwarnings('ignore')

import omicverse as ov
import lazyslide as zs
ov.utils.ov_plot_set()

adata, wsi = ov.space.histo.load_breast()
ov.space.histo.tile(wsi, tile_px=224, mpp=0.5)

## Extract GigaPath features (1536-d, gated)

GigaPath weights live behind HuggingFace gating. Request access at https://huggingface.co/prov-gigapath/prov-gigapath, then `huggingface-cli login` on this machine. The cell raises `GatedRepoError` otherwise.

In [ ]:
try:
    ov.space.histo.embed(wsi, model='gigapath',
                         batch_size=16, num_workers=0)
except Exception as e:
    print(f'gigapath unavailable: {type(e).__name__}: {e}')
    raise
wsi.tables['gigapath_tiles']

## Zero-shot prediction across 17 organs and 38,984 genes

In [ ]:
pred = ov.space.histo.predict_expression(
    wsi, method='stpath',
    organ='Breast', tech='Visium',
    genes=['EPCAM', 'ERBB2', 'CD68', 'ACTA2', 'VIM'],
)
pred

## Visualise

In [ ]:
import scanpy as sc
sc.pl.embedding(pred, basis='spatial',
                color=['EPCAM', 'ERBB2', 'CD68', 'ACTA2'],
                cmap='magma', s=12, ncols=2, frameon=False)

STPath returns log1p-normalised expression. For downstream analysis (`ov.space.svg`, `ov.pl.spatial`, …) treat `wsi.tables['stpath_tiles']` like any other Visium AnnData.